<h1 align="center">LNR_LAB1_Activity_2. PoS Tagging with HMM </h1>


---


<h4 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h4>
<h4 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h4>
<h4 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h4>
<br>

## **Installing necessary packages using pip**

In [6]:
import numpy
import re
import nltk
import sklearn
import nltk.tag as tagger
from nltk.tag import hmm

## **Downloading  the NLTK resources using the download method**

In [7]:
# brown corpus
nltk.download('brown')
from nltk.corpus import brown

# punkt tokenizer
nltk.download('punkt_tab')
# or
# nltk.download('punkt')
# it depends on your nltk version


[nltk_data] Downloading package brown to /home/fmargom/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fmargom/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [8]:
print("raw corpus:")
print(brown.raw()[:500])

raw corpus:


	The/at Fulton/np-tl County/nn-tl Grand/jj-tl Jury/nn-tl said/vbd Friday/nr an/at investigation/nn of/in Atlanta's/np$ recent/jj primary/nn election/nn produced/vbd ``/`` no/at evidence/nn ''/'' that/cs any/dti irregularities/nns took/vbd place/nn ./.


	The/at jury/nn further/rbr said/vbd in/in term-end/nn presentments/nns that/cs the/at City/nn-tl Executive/jj-tl Committee/nn-tl ,/, which/wdt had/hvd over-all/jj charge/nn of/in the/at election/nn ,/, ``/`` deserves/vbz the/at praise/nn and/c


# **PoS tagging with HMM using NLTK**

Using the [**HiddenMarkovModelTagger**](https://tedboy.github.io/nlps/generated/generated/nltk.tag.HiddenMarkovModelTagger.html) Object

## **Simplifying the Brown corpus**

Apply the following criterion:

- All tags will be simplified to keep only the grammatical category information.
- Remove the prefix "FW-" of a tag
- Split the PoS tag by symbols '-', '+' , '$' and '*' keeping the left part of the tag. For instance "NP+IN" will be simplified as "NP"
- Notice the specific tag “--” denoting a dash symbol.

In [9]:
def simplify_tag(tag):
    if tag == "--":
        return tag

    if tag.startswith("FW-"):
        tag = tag[3:]

    # Keep only the left-most grammatical category.
    return re.split(r"[-+\$*]", tag, maxsplit=1)[0]


def simplify_corpus(corpus, rule):
    simplifiedCorpus = []
    for sentence in corpus:
        simplified_sentence = [(word, rule(tag)) for word, tag in sentence]
        simplifiedCorpus.append(simplified_sentence)

    return simplifiedCorpus


simplifiedBrown = simplify_corpus(brown.tagged_sents(), simplify_tag)

nwords = 0 # number of words
voc, dtags = set(), set() # vocabulary, different tags
for s in simplifiedBrown:
    ws = [w for w, _ in s]
    ts = [t for _, t in s]
    nwords += len(ws)
    voc.update(ws)
    dtags.update(ts)

print(f"Number of words: {nwords}")
print(f"Vocabulary size: {len(voc)}")
print(f"Number of different tags: {len(dtags)}")
print(dtags)
print("First sentence with simplified tags:")
simplifiedBrown[0]

Number of words: 1161192
Vocabulary size: 56057
Number of different tags: 80
{'', 'DTS', 'HVZ', 'AP', "'", 'NRS', 'PPO', ')', 'RP', 'BEG', 'BE', 'PP', 'DOZ', 'NP', 'DTI', 'EX', 'NPS', 'WQL', 'NN', 'DO', ',', 'HVN', 'PN', 'QL', 'BED', '(', 'VBG', 'CD', 'UH', 'RBT', 'CS', 'WPO', ':', 'PPL', 'HV', 'DOD', 'NNS', 'OD', 'PPSS', 'WPS', 'VBN', 'ABX', 'PPLS', 'BER', 'DTX', 'MD', 'WDT', 'RN', '--', 'BEDZ', 'RBR', 'JJR', '``', 'DT', "''", 'VB', 'ABN', 'VBD', '.', 'QLP', 'HVD', 'JJS', 'TO', 'WRB', 'ABL', 'AT', 'JJ', 'BEN', 'CC', 'BEM', 'WP', 'HVG', 'JJT', 'RB', 'BEZ', 'NR', 'IN', 'NIL', 'PPS', 'VBZ'}
First sentence with simplified tags:


[('The', 'AT'),
 ('Fulton', 'NP'),
 ('County', 'NN'),
 ('Grand', 'JJ'),
 ('Jury', 'NN'),
 ('said', 'VBD'),
 ('Friday', 'NR'),
 ('an', 'AT'),
 ('investigation', 'NN'),
 ('of', 'IN'),
 ("Atlanta's", 'NP'),
 ('recent', 'JJ'),
 ('primary', 'NN'),
 ('election', 'NN'),
 ('produced', 'VBD'),
 ('``', '``'),
 ('no', 'AT'),
 ('evidence', 'NN'),
 ("''", "''"),
 ('that', 'CS'),
 ('any', 'DTI'),
 ('irregularities', 'NNS'),
 ('took', 'VBD'),
 ('place', 'NN'),
 ('.', '.')]

## **Splitting a corpus in TRAIN and TEST partitions using SKLEARN**

In [10]:
from sklearn.model_selection import train_test_split

# Split the simplifiedBrown corpus, the transformed corpus with the simplified labels: 90% for training and 10% for testing.
train_part, test_part = train_test_split(simplifiedBrown, test_size=0.1, shuffle=False)

## **Training the corpus using the train partition**

In [11]:
trainer = hmm.HiddenMarkovModelTrainer()
newtagger = trainer.train_supervised(train_part)

## **Evaluating the model using the test partition**

In [12]:
# This method evaluates a tagged-sequence in terms of accuracy
# How the predicted labels match the ground truth labels in the corpus
acc = newtagger.accuracy(test_part)
print(f"Accuracy: {acc:0.5}")



/home/fmargom/Documentos/uni/25-26/lnr/.venv/lib/python3.12/site-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
/home/fmargom/Documentos/uni/25-26/lnr/.venv/lib/python3.12/site-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/home/fmargom/Documentos/uni/25-26/lnr/.venv/lib/python3.12/site-packages/nltk/tag/hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)
/home/fmargom/Documentos/uni/25-26/lnr/.venv/lib/python3.12/site-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


Accuracy: 0.68701


In [13]:
# Have a look to the confusion matrix
print(newtagger.confusion(test_part))

     |                                                                                    B                                                                                                                                                                                             P              P                                                                                                          |
     |                                                 A    A    A                   B    E    B    B    B    B    B                        D    D         D    D    D              H    H    H    H              J    J    J              N         N         N                   P    P    P    P    P         Q         R    R                             V    V    V    V    W         W    W    W    W      |
     |              '                   -              B    B    B    A    A    B    E    D    E    E    E    E    E    C    C    C    D    O    O    D    T    T    T    E    H    V    V    V 

In [14]:
# Calculate the accuracy per tag
from collections import defaultdict

correct_by_tag = defaultdict(int)
total_by_tag = defaultdict(int)

for sentence in test_part:
    words = [word for word, _ in sentence]
    predicted = newtagger.tag(words)

    for (_, gold_tag), (_, pred_tag) in zip(sentence, predicted):
        total_by_tag[gold_tag] += 1
        if gold_tag == pred_tag:
            correct_by_tag[gold_tag] += 1

print(f"{'Tag':<10} | {'Accuracy':>10} | {'Instances':>9}")
print("-" * 36)
for tag in sorted(total_by_tag, key=lambda t: (correct_by_tag[t] / total_by_tag[t], total_by_tag[t])):
    acc = correct_by_tag[tag] / total_by_tag[tag]
    print(f"{tag:<10} | {acc:>9.2%} | {total_by_tag[tag]:>9}")

Tag        |   Accuracy | Instances
------------------------------------
WQL        |     0.00% |         1
RN         |     0.00% |         2
NPS        |     8.70% |        69
RBT        |    20.00% |         5
JJS        |    41.18% |        17
NP         |    44.59% |      3189
HVN        |    48.48% |        33
NRS        |    50.00% |         2
DTX        |    50.00% |         2
)          |    50.00% |        52
JJT        |    51.56% |        64
WP         |    52.94% |        17
VBZ        |    54.55% |       264
JJR        |    55.56% |       126
(          |    55.77% |        52
VBN        |    55.87% |      1899
WPS        |    56.20% |       274
NNS        |    56.73% |      3030
QL         |    56.85% |       693
RBR        |    58.10% |       105
'          |    58.54% |        41
JJ         |    58.68% |      4579
HVG        |    58.82% |        34
ABL        |    60.00% |        30
AP         |    60.00% |       625
NN         |    60.63% |     10876
NR         |    6

## **Searching wrong sentences**

In [15]:
def show_wrong_sentences(tagger, tagged_sentences, limit=5):
    shown = 0
    for sentence in tagged_sentences:
        words = [word for word, _ in sentence]
        predicted = tagger.tag(words)

        if predicted != sentence:
            print("Sentence:", " ".join(words))
            print("Gold tags :", sentence)
            print("Pred tags :", predicted)
            print()
            shown += 1

        if shown == limit:
            break


show_wrong_sentences(newtagger, test_part, limit=5)

Sentence: The physician led the horses to the stable after a cursory glance at the cringing slave .
Gold tags : [('The', 'AT'), ('physician', 'NN'), ('led', 'VBD'), ('the', 'AT'), ('horses', 'NNS'), ('to', 'IN'), ('the', 'AT'), ('stable', 'NN'), ('after', 'CS'), ('a', 'AT'), ('cursory', 'JJ'), ('glance', 'NN'), ('at', 'IN'), ('the', 'AT'), ('cringing', 'VBG'), ('slave', 'NN'), ('.', '.')]
Pred tags : [('The', 'AT'), ('physician', 'NN'), ('led', 'VBD'), ('the', 'AT'), ('horses', 'NNS'), ('to', 'IN'), ('the', 'AT'), ('stable', 'NN'), ('after', 'IN'), ('a', 'AT'), ('cursory', 'JJ'), ('glance', 'NN'), ('at', 'IN'), ('the', 'AT'), ('cringing', 'VBG'), ('slave', 'NN'), ('.', '.')]

Sentence: Had Dandy been older or wiser , instinct might have warned him that he would be well advised to flee from the Lalauries' tender care if he valued his life .
Gold tags : [('Had', 'HVD'), ('Dandy', 'NP'), ('been', 'BEN'), ('older', 'JJR'), ('or', 'CC'), ('wiser', 'JJR'), (',', ','), ('instinct', 'NN'), ('m